# Lección 3: Elementos básicos de Spark (RDD, Transformaciones y Acciones)  
## Proyecto: Retail Analytics Pipeline – RetailMax

## 1. Introducción

En esta lección se trabaja con RDDs (Resilient Distributed Datasets), la estructura fundamental de Apache Spark para el procesamiento distribuido de datos.

Se aplican transformaciones y acciones sobre datos reales del dominio e-commerce, permitiendo comprender cómo Spark construye y ejecuta el flujo de procesamiento mediante un DAG (Directed Acyclic Graph).

El objetivo es manipular grandes volúmenes de datos de manera eficiente y entender el linaje de las operaciones.

## Carga de datos

In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("RetailAnalyticsPipeline") \
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext

# Cargar dataset
file_path = "data/ecommerce_client_data.csv"

rdd = sc.textFile(file_path)

# Separar encabezado
header = rdd.first()
data = rdd.filter(lambda x: x != header)

## Transformación a estructura de columnas

In [3]:
# Separar columnas
rdd_split = data.map(lambda x: x.split(","))

print(rdd_split.take(3))

[['0', '0', '0', '0', '1', '0', '0.2', '0.2', '0', '0', 'Feb', '1', '1', '1', '1', 'Returning_Visitor', 'FALSE', 'FALSE'], ['0', '0', '0', '0', '2', '64', '0', '0.1', '0', '0', 'Feb', '2', '2', '1', '2', 'Returning_Visitor', 'FALSE', 'FALSE'], ['0', '0', '0', '0', '1', '0', '0.2', '0.2', '0', '0', 'Feb', '4', '1', '9', '3', 'Returning_Visitor', 'FALSE', 'FALSE']]


## Creación de Pair RDDs
Se generan estructuras clave-valor para facilitar agregaciones.

In [ ]:
pair_rdd = rdd_split.map(lambda x: (x[15], float(x[6])))

print(pair_rdd.take(5))

[('Returning_Visitor', 0.2), ('Returning_Visitor', 0.0), ('Returning_Visitor', 0.2), ('Returning_Visitor', 0.05), ('Returning_Visitor', 0.02)]


In [ ]:
# map
mapped = pair_rdd.map(lambda x: (x[0], x[1] * 1.1))

# filter
filtered = pair_rdd.filter(lambda x: x[1] > 50)

# flatMap
words_rdd = data.flatMap(lambda x: x.split(","))

# distinct
distinct_clients = pair_rdd.map(lambda x: x[0]).distinct()

# sortBy
sorted_rdd = pair_rdd.sortBy(lambda x: x[1], ascending=False)

print("Ejemplo filter:", filtered.take(3))
print("Ejemplo sorted:", sorted_rdd.take(3))

Ejemplo filter: []
